# RepairScope-X — Notebook 11
## Dream-7B Own Natural Failure Collection

Notebook 10 produced a real diffusion repair signal:

- **144/144** DLM repair trials completed;
- **44 task-successful repairs**;
- task success occurred on **all 3 datasets** and **all 4 repair scopes**;
- best fixed action = `global`, task success **38.89%**;
- per-instance oracle task success = **41.67%**;
- DLM Oracle Repair Headroom = **2.78 percentage points**;
- selective token preservation remained mechanically valid;
- `DLM_SURFACE_PILOT_READY = true`.

The strongest additional phenomenon was **error recurrence**: among still-failed repairs, `open_last_25` returned to the original wrong numeric state in **48%** of cases, whereas `global` did so in **0%**.

### Critical scientific issue addressed here

Notebook 10 used **Qwen-generated natural failures** as the starting failed responses. That proves Dream can perform DLM-native repair, but it is not yet a clean test of **Dream repairing its own failures**.

This notebook fixes that.

It runs Dream itself from scratch on the same balanced 48 calibration problems:

- 16 Countdown
- 16 GSM8K
- 16 SVAMP

using two independent stochastic seeds:

**48 problems × 2 Dream generations = 96 generations**

Every output is evaluated with the same three-level protocol used in Notebook 10:

1. strict `FINAL_ANSWER` success;
2. numeric success;
3. task-aware success.

Every **task failure** becomes a candidate for the next self-repair experiment.

### Quantization status

This is still the exploratory `bnb_nf4_4bit` Dream condition needed for the available T4-class GPU. It is useful for method development, but final-paper evidence still needs at least one full-precision DLM replication.

Embedded source payload SHA-256: `f7ca43a0318c5664ce1bd389335b785a3c48b0025385f9126a1695ed0cea37f4`

In [ ]:
!pip -q install -U "transformers>=4.57,<5" "accelerate>=1.0" "bitsandbytes>=0.47" pandas

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3300
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 11 requires a CUDA GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError(
        "This exploratory 4-bit Dream run expects a T4-class GPU (~15 GB) or larger."
    )

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover and validate the exact 48-source benchmark

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "f7ca43a0318c5664ce1bd389335b785a3c48b0025385f9126a1695ed0cea37f4"

_raw = zlib.decompress(
    base64.b64decode('''eNq1W22P20aS/isNwQd/0czxXeTgcAtn48AxkmzgZBHkDgdti2xJhCi2li8jy0H++z1V3ZRISZORlGgAIza7q1hVXfXUU83Jb6Nat1Wq6tHT//422lR6Vqj1NM9GT6NUt2WT6W35lMoin1WyyXX55MZO+OQ5o/Eok42sVdPfOZ1Xql5ibSOrJqf9tHqQppVKrzck9M9aiWapRNmuZ6qqhTcZCy8aC1lmwguF+izTptgJXabYpwXksjY1Io2sFqoRXvwoPqmmrUohxTPekglZ5c1yrZo8hYINjKnxVlap8GeLRfGvb7794d1303c//PTL+09P4r/M+//7X48wzigePUGz8/v4iniEf3083ATxSGw8gkvi4Qb3igc0XxUP3/3r4+E7iIdj4uE6l8QjcO4VD2i+Kh7uHeol7JVLfEk44uhu9UKqrwpIcIeC8cdcM2R8clG5+HcrF/+6aER3QA9UimvTw3UvQ4/kfvCRXFkvd8APBMO18HFRubj3ay/utf0lmNyh346pYsj2i5qtfzcw9a8EUz/+64MRIDkCC6YXgceD694rHqT6OvaR3CE7kB5eaCNyUbd98O6Gp6T6uhS5R0Sov0wsovoXlczkbghCqq9D1DsgiNuD1MsoiH83hupfyVDDuwCqF1wTjfBu0QivjIZ3B0hNDuVyGYCE3t1aDKm+KiB3iAf1l321XIQfd+swV/aX4B7TPtIjuYqN3Y+MnYnGol7Hq6emknn5FLnhMB148TK3P+pabZZiKWuYXal5lS8UdukKroutbFQllor/k5dimddiqdtambjIUqhCpU0FB/WzKhEO2YhUl3W7VrXY6C3E8CTL53NVqbIR0KzqR/EzhU03shByTWcl9Fysdal2RoO1aSN3tZjriqOsSlUtdgLvzsRsx4/61gpY1iwrBb05vZvWrW78baDJmNVX1HfzUXw7Ny8ceIbtnUNbXeHg3oSOQyGRZHizHItmmyMhti/Zb2SP38Z6xwJnk7YFHpiE6kfmVeuRtguZIVHqQc7AvFeyZuJH4a1Z800lyxXnwFd5USB7npV4E3j8BDbtxEy3iyXaqSjIILHJv3xBhnHKkxd82I/ivUyXZg1ZUzfiDQZBUmHUbyRqiPyWeIU5XKPn0W5Y0GtNJtScQ5SfNo80W8anaU3MxBvfga5KyWw3Rh5vxbrF+41ApnHAB2dKvf1bv3knrwXTdYJbg/lONFWbrpAFJf5U1Y6SSn7O1+2anHL9MeXaalHTv+p2sylyKqJf4bSqMglfOGtYxxZRLrTMlMUXF5Iz/ZkrApm5VfnCPCR1CuHnYizHIm/OyIqUCxZZnSNQRjgayGp+/1BYLhARoyJ0RC3T1VADHq7yQi8que70fKDDkCXyRkI3b8vLhQiOd3KMtkrUDZ0UvY9qcO9+/8jc1/I/nHjRrUdGAMZP6Uh6WLPHsb/LSo7FR1mqxmDlR0UHy7mFJA2ewqdoDzUvQaE6CLyZhMcwUei65vcY/fQqBB1hWcsVZOZGQarXs7xkibw6FMesRTNDES1wuiReK64xtSa8jp3/6PzCPjqIDYBQ9fHltXqIYvfm4L5/VgjVWlclvfoDV0QDn8BQQrHOy5ZS0vqQarQWZboRJY8qDAYxGHPKFxpKsJvK3iCo3FBboi6M5ZkCwgBCNb2LckkTwK9MSrIsIwNb8bbeW1XptuGwwq5+1gXha1nnR8mtgfleNo3t1BslEQexKWTZ8D7jLDKFsAIhaag9dH9/FMIV9b9bCU/nWnOSGQ01VxRnDMrSFNphEXJIUVS3Cb93uoPiZaSPl3A8TWMD7QIyTlco50xjewOo2eMxx5t6BKmdw3eG9YOnA1x2vfD1Op9Mbu5ztuvO5ToH/atUW2LHWPyomBHVXX24ERpzW2bs/wxF9oBUSoF5K2UI31IWc3NEtsyJTcn1rIUfFcX5g0J2mSy0qe1Rsqme3iUOLtMLhvJz0rKo9VBFrVBKgJY6z/YdgBDC4LC1qQ9e3StIZQfJBwOQOplRYJDhqg4ZRZOb6cZHSZwOb6xFJJqHeplXjcmckKFqDnt/JrQzPYKpBM1qxglOKlAJQ6X6Rkev5U6U+P7tbb1Ol1oXqLA5pUsuDYtcVBRQAjIzS6Rar0CTarkopQHjVDeNZNZkyfLOFAP17W7j2LZUqESmHM5ooJ3CMT5R2Rf1/0h036AOHhD6MPkkaHlBlFEUHc1yLMqh/uv5LJrOqWHLfp2zHl28X3ceLZB/qSrNoEfumyctbBT1RsEtahRtSdQHBQ9b4UvQ9Rwzt7DIM4pAcgnTiYn3+PfD93oGEVNlWZU/U79yOj3UcjA5VT2qYxbmMBwmsUVNp55tJDtowChZ0BR9c2wAifHL+nF8FRInbpL8GUhU3KFnOXLygyqrnYHyMVUYQTXx9jexgS+DVYfnNDzDhMawC2jgJDNaaIjBYHW6Z3zaHWjk7hQyeaEsN1Ixy9SDzAqCV0PiT26foZclyAczFA+5jgkYXW4tQeY1huGxgaU9qw0FiiWlfPrAvKyE3cSRaZxGjy7sFURHwfZ6TDIehulZpbclM+Yxs5sdI2KmipyO51H8QqopglWu5uhehZLPdjRe5U1KqxRpNCnDi5boW2NutkXeNAUJas7NnxocOLGl2o7iTY6ThZcN/sr1D8k9Y9xbBfz9xZKSA8vlKIATwxcThXEnWKk1DoICdBLB6xhW6N18kr/Ighr7nBKKqQf7SxYviYE0CDDPZLVeoxABi3NdZGQyZR9LbSSaz9fyma5/5qwL0yvPMmJNGFpzvhOl4HfQdIsazzI2THZzrikde5NBeKI2sMBy1SVOh5s8lQED77gbvPdGIpuwE8dW4uBr5BewDZSVaYc9Ab6q2DtF85sxe8iwoteiHQTen7lFSPP6bUevCE090a7aQhWUGIFYtDkMqe11k0hV1RDqHQrkOdeIbm2vEqwoKwowJFYIU7dmVPFSdLRklBxLDYazwxvtOpEmewYHgwfmWtN43HM6sV5P7NYZ0U5DMYT0V68AkyD8E1eAS0OufItEezfBYZWqgdHdVY1K1cmqIVx0W0BwsN8LTeZEneH22o7s6z7JzDT6WVW/xNNe5Za3e/8Tzgrdliyt88+26PAPY5akayy6DqZlXe6rN20rwmCbtKZoa8b6A+dZ5kWGTfuWdXKm526562e53li3UjD0hzAZXizxhstc+7qtOpx4V61kWVt++bP6TEYrqgowa97wSX7R1YzOzPJrUS/1Bp5kSrwRHibvmWnzpoN1JPxxAPGcCEgP7nd2ywC7/Us8dp3oVo9B0AB7BH0RWlC9Uk03kSL3unI3C3xwmNsqIKNdN7NqDFBV1JQ7oX2SDrZWqrt66l6E3qHhJkzo+5xEl/gcTJJbfe5IxJY4AbMhc6Ld9c5M0ZEtZfVMF4YZs44503VqCNxe7FUT7xC+a+/sELgq36jBPZEXHBbbcrBMFDeTu168XtrJI+Thesv0ySAkaTPoGkuGo0HsXBLHOJ7cXC2y5EmloIt/hE0zB0rlpiNqldoKuZU75JalLGj+RH1qYWII2KpWNFxRC+qCHDp9VaKEEhJ5FD8gTEvTq6LhHvulxZoCM/tTAzGA/mZCzIxNn+fUJHuW2JFB12zOwYFBSQYXpefNFfk/KBcJbk9m+qHYtPXyod2YZPJFiiGHywkuL3ZrkWK8pTslziNDR0gwtMPEXnqGzI0nYLTYvdfB7n4x7zuO2F6SN+1FSPneQj24z/Augir35nQzTVIbHAJBmBf0ZYcjkTc8alokC5zBIvORx/71TGPQqGEJmhiL4tWL8BNPfGfy50EXHKA3nmi9Mucc9Z6u9XO+pwDiLX1h+AKkACLsuvuSt4Cwiq8+0Gt3ujVXt26wH4iMCtK8lTTHYDHZjx/81uNyoWd88laWcadTPRidL2rJt4N1L1YYU2Up6c4HwfgRoJFv3tb22aD2LRHtdpOwrhayzL8o+vwByKcpYGNoGx6CQ+/9tyv7/PjbK3f4J74mkX+rr+8Y1zE28oBEXzyrBieipAX/44b1VZuusHUp14CvAiOLrYIxyAlOuaABtirpkoZ6FvrFc17nja7oDnvNV2lntQhtcm1TKVDutjZNCkGNvQRNS2+ApKxJZS8ooE9ZNmF5vIttq/pwoPLWkD/W093SZ0MInsQXYbCXeLeexEe6PWS+E2PupA/IHY3houD5HgRiaWiw61vM7O/5yF9T7U7elZxhSJ0IU96P3dfU16j8abtxb+a8PMzQx17M27Ka2Tu8k3bKn88a4n/dLp7guWPHvSY9BJJur51SCl2roybLGvr+xpf4O3Hcm9sIX8Uho8SnnbnFpCvZiO8Ma8o4LhqqQFUuihzBYRp3Zh3zSsmz3tBns7GbYVj/WbXdRR/vPKN3QDouQp745qCgDx3Ter6xYYcG7P6Ux3d4a2Fa2gsA+hAf7qUGEn20/UPVVzeaJLi50XzXrugXQ3LCSzegz1M53dXBHh7OzH09nJI8AnZj7aaQOxIIJ2ZHbbZgfH/Ozc4+8WCNthbqVB/cpY2DSRfYdRGfSsKbCx9gjjawIZbEjaXS2+5iLoqGS8t81X3tZTrBefoovOisBruNp5RSn5LxP5DpvhZzbRzrGbDNy8jmzQXxK6gOgaIX2d+woF/OOL5spg9RfdpFAsj5FCHs+Fbodbv2VPQQjW6FEUCZSXtx9ibr/8ajUjeKeJnrTBeyUaOn30bIMlnUfFk/HrVl/u9WTfkZ4tE9hu+rad2mKZj/vC2mldrIvKppjhmP6Jopbc4u+xFe2a7BK9Oz61G4j2w9pehM+2+C/HgkuXecXcW7ZxhYp/P8s8qmZiPdQhV6JovRYHEo6Dz68fAHqjAmp4U62Rm4Uf9nAoOL9dRuNo5MwbGySus17Xe8Sf8nDhCCtUImlTmyZEq/j6iqZ06VqV6NkGmtwo68pF88IieQHdNyjqOJYBRypWzyL9K6NitnWAumwSxvpurzptD8a3g7OPv1d99Pf/rnp2/e/f399Mdvv/vHz9NP7999/at5we+DkzeRgpNrdNYd/w+ep8Hr0oKOcHjC04ozB67G/uDn5Kz3O8OjoAwzar/t6FQSipwsj4Ly6NjH/VCStBsmnhO5oedGiRc4sd22xWA2rVWqS0pnz310PX8SJc7E9aOJgwRcy89TNI/V9LmS6+lixmzJibwwdIMkDlzfSQgl9iHSaMZTTMzNlP/3zgsCNfSe3H8pUKfZdj5QTtj/ia4LVBx6fuI7gTdxHSfx/POBch4jZO8kCV0v9BGMc4EKH4PAdeMg9JyJ5wZ+9EKgJhcG6tSvFwN19PNCoE4jf02gQnfiO14Qx1488ZKX8slx3MSdOJRUcRKdD1PsuXEYB07kRFGchC+EKXQuDJN/aeFdmE9eeF1gIj/yvdgJfM9J3NhNXsogVE8URFEQYK/nng9N5KNkvSgMg3iC+ITUprJKYX2tM1WYrvw1PXjA/P2f5m/PzsO3ZU2/fdg8TL4aDSVo4qxNbJ3Q94N0lsylnE9gdeJl6TzJ4ok/SaPAmylv5slIqtHv/w+0eGDx''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))

SOURCES = UPSTREAM["sources"]
UPSTREAM_DLM_GATE = UPSTREAM["notebook10_gate"]

assert len(SOURCES) == 48
assert len({x["problem_id"] for x in SOURCES}) == 48
assert Counter(x["dataset"] for x in SOURCES) == {
    "countdown_fresh": 16,
    "gsm8k": 16,
    "svamp": 16,
}
assert all(x["partition"] == "calibration" for x in SOURCES)
assert UPSTREAM_DLM_GATE["DLM_SURFACE_PILOT_READY"] is True

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Sources:", len(SOURCES))
print("Dataset balance:", dict(Counter(x["dataset"] for x in SOURCES)))
print("Source payload validation: PASS")

## 3. Strict and numeric verifiers

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)


def close_num(a, b):
    return (
        a is not None
        and math.isclose(
            float(a),
            float(b),
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )


def strict_final(text):
    vals = FINAL_RE.findall(str(text))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def last_number(text):
    vals = NUM_RE.findall(
        str(text).replace(",", "")
    )

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def basic_verification(text, target):
    strict = strict_final(text)
    loose = last_number(text)
    target = float(target)

    return {
        "strict_prediction": strict,
        "numeric_prediction": loose,
        "target": target,
        "strict_success": bool(close_num(strict, target)),
        "numeric_success": bool(close_num(loose, target)),
    }


assert basic_verification(
    "reasoning\nFINAL_ANSWER: 4",
    4,
)["strict_success"]

assert basic_verification(
    "The result is 4.",
    4,
)["numeric_success"]

print("Strict/numeric verifier tests: PASS")

## 4. Countdown structured verifier

In [ ]:
COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+"
    r"to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.IGNORECASE,
)

ARITH_RUN_RE = re.compile(
    r"[0-9eE+\-*/().=\s]{3,}"
)

ALLOWED_BINOPS = (
    pyast.Add,
    pyast.Sub,
    pyast.Mult,
    pyast.Div,
)

ALLOWED_UNARY = (
    pyast.UAdd,
    pyast.USub,
)


def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(
        str(prompt)
    )

    if m is None:
        raise ValueError(
            "Countdown prompt format was not recognized."
        )

    source_nums = [
        float(x)
        for x in NUM_RE.findall(
            m.group(1)
        )
    ]

    target = float(
        m.group(2)
    )

    return source_nums, target


def safe_arith_tree_value_and_literals(expr):
    try:
        tree = pyast.parse(
            expr,
            mode="eval",
        )
    except Exception:
        return None

    literals = []

    def walk(node):
        if isinstance(node, pyast.Expression):
            return walk(node.body)

        if isinstance(node, pyast.Constant):
            if not isinstance(node.value, (int, float)):
                raise ValueError

            val = float(node.value)

            if not math.isfinite(val):
                raise ValueError

            literals.append(val)
            return val

        if isinstance(node, pyast.BinOp) and isinstance(node.op, ALLOWED_BINOPS):
            a = walk(node.left)
            b = walk(node.right)

            if isinstance(node.op, pyast.Add):
                return a + b

            if isinstance(node.op, pyast.Sub):
                return a - b

            if isinstance(node.op, pyast.Mult):
                return a * b

            if isinstance(node.op, pyast.Div):
                if abs(b) < 1e-15:
                    raise ZeroDivisionError
                return a / b

        if isinstance(node, pyast.UnaryOp) and isinstance(node.op, ALLOWED_UNARY):
            value = walk(node.operand)

            if isinstance(node.op, pyast.USub):
                return -value

            return value

        raise ValueError

    try:
        value = walk(tree)

        if not math.isfinite(value):
            return None

        return {
            "value": float(value),
            "literals": literals,
        }

    except Exception:
        return None


def number_multiset_equal(a, b):
    if len(a) != len(b):
        return False

    aa = sorted(float(x) for x in a)
    bb = sorted(float(x) for x in b)

    return all(
        close_num(x, y)
        for x, y in zip(aa, bb)
    )


def countdown_candidates(text):
    text = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )

    candidates = []

    for m in ARITH_RUN_RE.finditer(text):
        run = m.group(0).strip()

        if not any(
            op in run
            for op in ["+", "-", "*", "/"]
        ):
            continue

        parts = [
            part.strip()
            for part in run.split("=")
            if part.strip()
        ]

        candidates.extend(parts)

    return sorted(
        set(candidates),
        key=len,
        reverse=True,
    )


def countdown_task_verification(text, prompt):
    source_nums, target = parse_countdown_prompt(prompt)

    checked = []

    for expr in countdown_candidates(text):
        parsed = safe_arith_tree_value_and_literals(expr)

        if parsed is None:
            continue

        uses_numbers_once = number_multiset_equal(
            parsed["literals"],
            source_nums,
        )

        value_correct = close_num(
            parsed["value"],
            target,
        )

        checked.append({
            "expression": expr,
            "value": parsed["value"],
            "literals": parsed["literals"],
            "uses_numbers_once": uses_numbers_once,
            "value_correct": value_correct,
        })

        if uses_numbers_once and value_correct:
            return {
                "task_success": True,
                "valid_expression": expr,
                "expression_value": parsed["value"],
                "source_numbers": source_nums,
                "target": target,
                "checked_candidates": checked,
            }

    return {
        "task_success": False,
        "valid_expression": None,
        "expression_value": None,
        "source_numbers": source_nums,
        "target": target,
        "checked_candidates": checked,
    }


_demo_prompt = (
    "Use the numbers 19, 29, and 24 exactly once "
    "to produce the target 14. "
    "Return a valid arithmetic expression."
)

assert countdown_task_verification(
    "The valid expression is 19 + 24 - 29 = 14.",
    _demo_prompt,
)["task_success"]

assert not countdown_task_verification(
    "29 - 19 = 10. FINAL_ANSWER: 14",
    _demo_prompt,
)["task_success"]

print("Countdown verifier regression tests: PASS")

## 5. Unified task verifier

In [ ]:
def verify_generation(text, source):
    basic = basic_verification(
        text,
        source["target"],
    )

    if source["dataset"] == "countdown_fresh":
        structured = countdown_task_verification(
            text,
            source["prompt"],
        )

        task_success = bool(
            structured["task_success"]
        )

    else:
        structured = None

        task_success = bool(
            basic["numeric_success"]
        )

    return {
        **basic,
        "task_success": task_success,
        "countdown_structured": structured,
    }


print("Unified verifier: READY")

## 6. Dream natural-generation configuration

In [ ]:
MODEL_ID = "Dream-org/Dream-v0-Instruct-7B"
MODEL_REVISION = "05334cb9faaf763692dcf9d8737c642be2b2a6ae"

GENERATION_SEEDS = [
    3301,
    3302,
]

DENOISING_STEPS = 64
MAX_NEW_TOKENS = 96
TEMPERATURE = 0.2
TOP_P = 0.95
ALG = "entropy"
ALG_TEMP = 0.0

EXPECTED_GENERATIONS = (
    len(SOURCES)
    * len(GENERATION_SEEDS)
)

assert EXPECTED_GENERATIONS == 96

print("Generation seeds:", GENERATION_SEEDS)
print("Expected Dream generations:", EXPECTED_GENERATIONS)

## 7. Load Dream-7B in exploratory NF4 mode

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

assert hasattr(
    model,
    "diffusion_generate",
)

MODEL_DEVICE = next(
    model.parameters()
).device

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Device:", MODEL_DEVICE)
print("Load seconds:", round(time.time() - t0, 2))
print(
    "Allocated GB:",
    round(torch.cuda.memory_allocated()/2**30, 3),
)

## 8. Prompt builder and forward-call counter

In [ ]:
SYSTEM_PROMPT = (
    "Solve the quantitative problem carefully. "
    "Give a mathematically correct answer. "
    "When possible, end with `FINAL_ANSWER: <number>`."
)


def encode_problem_prompt(problem):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": problem,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    return {
        "input_ids":
            encoded.input_ids.to(MODEL_DEVICE),
        "attention_mask":
            encoded.attention_mask.to(MODEL_DEVICE),
    }


class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0

        self.handle = (
            self.module
            .register_forward_pre_hook(
                self._hook
            )
        )

        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


print("Generation helpers: READY")

## 9. Dream from-scratch generation runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_dream_generation(source, seed):
    reseed(seed)

    prompt = encode_problem_prompt(
        source["prompt"]
    )

    prompt_len = int(
        prompt["input_ids"].shape[1]
    )

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        output = model.diffusion_generate(
            prompt["input_ids"],
            attention_mask=
                prompt["attention_mask"],
            max_new_tokens=
                MAX_NEW_TOKENS,
            output_history=False,
            return_dict_in_generate=True,
            steps=
                DENOISING_STEPS,
            temperature=
                TEMPERATURE,
            top_p=
                TOP_P,
            alg=
                ALG,
            alg_temp=
                ALG_TEMP,
        )

    wall = time.time() - t0

    new_ids = output.sequences[
        0,
        prompt_len:
        prompt_len + MAX_NEW_TOKENS
    ].detach()

    text = tokenizer.decode(
        new_ids.tolist(),
        skip_special_tokens=True,
    ).strip()

    verification = verify_generation(
        text,
        source,
    )

    return {
        "generation_id":
            source["problem_id"]
            + "|dream|"
            + str(seed),
        "problem_id":
            source["problem_id"],
        "dataset":
            source["dataset"],
        "partition":
            source["partition"],
        "prompt":
            source["prompt"],
        "target":
            source["target"],
        "seed":
            int(seed),
        "output":
            text,
        "verification":
            verification,
        "strict_success":
            bool(
                verification["strict_success"]
            ),
        "numeric_success":
            bool(
                verification["numeric_success"]
            ),
        "task_success":
            bool(
                verification["task_success"]
            ),
        "actual_forward_calls":
            int(fc.calls),
        "generated_canvas_tokens":
            int(new_ids.numel()),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 10. One real Dream generation smoke test

In [ ]:
smoke = run_dream_generation(
    SOURCES[0],
    GENERATION_SEEDS[0],
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "output",
                "verification",
                "prompt",
            }
        },
        indent=2,
    )
)

print("\nDREAM OUTPUT\n")
print(smoke["output"])

assert smoke["actual_forward_calls"] >= DENOISING_STEPS
assert smoke["generated_canvas_tokens"] == MAX_NEW_TOKENS

print("\nDREAM NATURAL-GENERATION SMOKE: PASS")

## 11. Output directory and immutable run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook11_Dream_Own_Failures"
    )

else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook11_Dream_Own_Failures"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

signature_payload = {
    "protocol_version":
        "dream-own-failures-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "quantization":
        "bnb_nf4_4bit_exploratory",
    "generation_seeds":
        GENERATION_SEEDS,
    "denoising_steps":
        DENOISING_STEPS,
    "max_new_tokens":
        MAX_NEW_TOKENS,
    "temperature":
        TEMPERATURE,
    "top_p":
        TOP_P,
    "alg":
        ALG,
    "alg_temp":
        ALG_TEMP,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

GENERATIONS_PATH = (
    OUT_DIR
    / "dream_natural_generations.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        GENERATIONS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get(
        "run_signature"
    ) != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 11 results use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 12. Resume-safe 96-generation Dream sweep

In [ ]:
def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from e

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found generation from another Notebook 11 configuration."
                )

            gid = rec["generation_id"]

            if gid in done:
                raise RuntimeError(
                    f"Duplicate generation_id: {gid}"
                )

            done[gid] = rec

    return done


EXPECTED_IDS = {
    source["problem_id"]
    + "|dream|"
    + str(seed)
    for source in SOURCES
    for seed in GENERATION_SEEDS
}

assert len(EXPECTED_IDS) == 96

completed = load_completed(
    GENERATIONS_PATH
)

print(
    "Already completed:",
    len(completed),
    "/ 96",
)

counter = 0

with open(
    GENERATIONS_PATH,
    "a",
    encoding="utf-8",
) as fp:

    for source in SOURCES:
        for seed in GENERATION_SEEDS:
            counter += 1

            gid = (
                source["problem_id"]
                + "|dream|"
                + str(seed)
            )

            if gid in completed:
                print(
                    f"[{counter}/96] SKIP {gid}"
                )
                continue

            try:
                rec = run_dream_generation(
                    source,
                    seed,
                )

                rec[
                    "run_signature"
                ] = RUN_SIGNATURE
                rec[
                    "status"
                ] = "ok"
                rec[
                    "error"
                ] = None

            except torch.cuda.OutOfMemoryError as e:
                torch.cuda.empty_cache()
                gc.collect()

                rec = {
                    "generation_id":
                        gid,
                    "problem_id":
                        source[
                            "problem_id"
                        ],
                    "dataset":
                        source[
                            "dataset"
                        ],
                    "seed":
                        seed,
                    "status":
                        "oom",
                    "error":
                        str(e),
                    "run_signature":
                        RUN_SIGNATURE,
                }

            except Exception as e:
                rec = {
                    "generation_id":
                        gid,
                    "problem_id":
                        source[
                            "problem_id"
                        ],
                    "dataset":
                        source[
                            "dataset"
                        ],
                    "seed":
                        seed,
                    "status":
                        "error",
                    "error":
                        f"{type(e).__name__}: {e}",
                    "run_signature":
                        RUN_SIGNATURE,
                }

            fp.write(
                json.dumps(
                    rec,
                    ensure_ascii=False,
                )
            )
            fp.write("\n")
            fp.flush()

            if rec["status"] == "ok":
                print(
                    f"[{counter}/96] {gid} | "
                    f"strict={rec['strict_success']} | "
                    f"numeric={rec['numeric_success']} | "
                    f"task={rec['task_success']} | "
                    f"NFE={rec['actual_forward_calls']}"
                )

            else:
                print(
                    f"[{counter}/96] {gid} | "
                    f"{rec['status']} | {rec['error']}"
                )

print("Dream natural-generation sweep finished.")

## 13. Validate the 96-generation matrix

In [ ]:
records = load_completed(
    GENERATIONS_PATH
)

missing = (
    EXPECTED_IDS
    - set(records)
)

extra = (
    set(records)
    - EXPECTED_IDS
)

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Generation sweep incomplete. Rerun the sweep cell; completed generations will be skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected generation IDs exist."
    )

assert len(records) == 96

rows = [
    records[gid]
    for gid in sorted(
        EXPECTED_IDS
    )
]

status_counts = Counter(
    x["status"]
    for x in rows
)

print(
    "Statuses:",
    dict(status_counts),
)

if status_counts.get("oom", 0):
    raise RuntimeError(
        "At least one Dream natural generation OOMed."
    )

if status_counts.get("error", 0):
    raise RuntimeError(
        "At least one Dream natural generation errored."
    )

GEN_DF = pd.DataFrame(
    rows
)

assert len(GEN_DF) == 96
assert GEN_DF[
    "generation_id"
].nunique() == 96

assert (
    GEN_DF[
        "actual_forward_calls"
    ] >= DENOISING_STEPS
).all()

print("96-generation Dream matrix integrity: PASS")

## 14. Natural-generation performance summary

In [ ]:
summary_rows = []

for dataset, group in GEN_DF.groupby(
    "dataset"
):
    summary_rows.append({
        "dataset":
            dataset,
        "generations":
            len(group),
        "strict_success_rate":
            float(
                group[
                    "strict_success"
                ].mean()
            ),
        "numeric_success_rate":
            float(
                group[
                    "numeric_success"
                ].mean()
            ),
        "task_success_rate":
            float(
                group[
                    "task_success"
                ].mean()
            ),
        "task_failures":
            int(
                (
                    ~group[
                        "task_success"
                    ]
                ).sum()
            ),
        "mean_actual_nfe":
            float(
                group[
                    "actual_forward_calls"
                ].mean()
            ),
        "mean_wall_seconds":
            float(
                group[
                    "wall_seconds"
                ].mean()
            ),
    })

SUMMARY_DF = pd.DataFrame(
    summary_rows
)

display(
    SUMMARY_DF
)

print(
    "Overall task success:",
    round(
        float(
            GEN_DF[
                "task_success"
            ].mean()
        ),
        4,
    ),
)

print(
    "Overall task failures:",
    int(
        (
            ~GEN_DF[
                "task_success"
            ]
        ).sum()
    ),
)

## 15. Extract Dream's own natural failures

A generation enters the failure set **only when task-aware verification fails**.

Each Dream failure receives a stable `failure_id = problem_id|dream|seed`.

In [ ]:
FAILURE_DF = GEN_DF[
    ~GEN_DF[
        "task_success"
    ]
].copy()

FAILURE_DF[
    "failure_id"
] = FAILURE_DF[
    "generation_id"
]

failure_records = (
    FAILURE_DF
    .to_dict(
        "records"
    )
)

print(
    "Dream natural failures:",
    len(
        failure_records
    ),
)

print(
    "Failure distribution:",
    dict(
        Counter(
            x["dataset"]
            for x in failure_records
        )
    ),
)

print(
    "Unique failed problem IDs:",
    FAILURE_DF[
        "problem_id"
    ].nunique(),
)

## 16. Failure persistence across independent Dream generations

A problem that fails under both Dream seeds is a stronger self-repair candidate than a one-off stochastic failure.

In [ ]:
problem_persistence_rows = []

for (
    pid,
    dataset,
), group in GEN_DF.groupby(
    [
        "problem_id",
        "dataset",
    ]
):
    failures = int(
        (
            ~group[
                "task_success"
            ]
        ).sum()
    )

    problem_persistence_rows.append({
        "problem_id":
            pid,
        "dataset":
            dataset,
        "generations":
            len(group),
        "failure_count":
            failures,
        "failure_rate":
            failures
            / len(group),
        "persistent_failure":
            bool(
                failures
                == len(group)
            ),
    })

PERSISTENCE_DF = pd.DataFrame(
    problem_persistence_rows
)

print(
    "Problems failing under both Dream seeds:",
    int(
        PERSISTENCE_DF[
            "persistent_failure"
        ].sum()
    ),
    "/ 48",
)

display(
    PERSISTENCE_DF.sort_values(
        [
            "persistent_failure",
            "failure_rate",
            "dataset",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
)

## 17. Self-repair readiness gate

The next self-repair experiment is ready when:

- all 96 Dream generations completed cleanly;
- at least 12 task failures exist;
- failures cover at least 2 datasets;
- at least 6 unique problem IDs fail;
- at least 3 problem IDs fail under both Dream seeds.

The gate is intentionally about obtaining a sufficiently diverse **Dream-origin failure corpus**, not about model accuracy.

In [ ]:
failure_count = int(
    len(
        FAILURE_DF
    )
)

datasets_with_failure = int(
    FAILURE_DF[
        "dataset"
    ].nunique()
)

unique_failed_problems = int(
    FAILURE_DF[
        "problem_id"
    ].nunique()
)

persistent_failed_problems = int(
    PERSISTENCE_DF[
        "persistent_failure"
    ].sum()
)

READY_GATE = {
    "generations":
        int(
            len(
                GEN_DF
            )
        ),
    "unique_generation_ids":
        int(
            GEN_DF[
                "generation_id"
            ].nunique()
        ),
    "task_failures":
        failure_count,
    "datasets_with_failure":
        datasets_with_failure,
    "unique_failed_problem_ids":
        unique_failed_problems,
    "persistent_failed_problem_ids":
        persistent_failed_problems,
    "minimum_actual_nfe":
        int(
            GEN_DF[
                "actual_forward_calls"
            ].min()
        ),
    "quantization":
        "bnb_nf4_4bit_exploratory",
}

READY_GATE[
    "DREAM_OWN_FAILURE_CORPUS_READY"
] = bool(
    READY_GATE[
        "generations"
    ] == 96
    and READY_GATE[
        "unique_generation_ids"
    ] == 96
    and READY_GATE[
        "minimum_actual_nfe"
    ] >= DENOISING_STEPS
    and READY_GATE[
        "task_failures"
    ] >= 12
    and READY_GATE[
        "datasets_with_failure"
    ] >= 2
    and READY_GATE[
        "unique_failed_problem_ids"
    ] >= 6
    and READY_GATE[
        "persistent_failed_problem_ids"
    ] >= 3
)

print(
    json.dumps(
        READY_GATE,
        indent=2,
    )
)

## 18. Save Dream-own failure artifacts

In [ ]:
summary_path = (
    OUT_DIR
    / "dream_natural_summary.csv"
)

failures_path = (
    OUT_DIR
    / "dream_own_failures.jsonl"
)

persistence_path = (
    OUT_DIR
    / "dream_failure_persistence.csv"
)

gate_path = (
    OUT_DIR
    / "DREAM_OWN_FAILURE_READY.json"
)

SUMMARY_DF.to_csv(
    summary_path,
    index=False,
)

PERSISTENCE_DF.to_csv(
    persistence_path,
    index=False,
)

with open(
    failures_path,
    "w",
    encoding="utf-8",
) as f:
    for rec in failure_records:
        f.write(
            json.dumps(
                rec,
                ensure_ascii=False,
            )
        )
        f.write("\n")

gate_path.write_text(
    json.dumps(
        READY_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 11 artifacts:",
    OUT_DIR,
)

## 19. Artifact manifest

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    GENERATIONS_PATH,
    summary_path,
    failures_path,
    persistence_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "files": {
        p.name:
            sha256_file(
                p
            )
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR
    / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 20. ZIP and download all Notebook 11 results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook11_Dream_Own_Failures_results"
)

zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=
        OUT_DIR,
)

print(
    "Created:",
    zip_path,
)

print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(
        zip_path
    )

# Progression rule

If:

`DREAM_OWN_FAILURE_CORPUS_READY = true`

the next notebook should use **only Dream-origin failures** and test whether the same Dream model can repair its own failed denoising trajectories under:

- local selective remasking;
- broader selective remasking;
- global regeneration;
- matched actual NFE;
- task-aware verification;
- preservation;
- same-error recurrence;
- per-instance repairability.

That self-repair experiment is substantially more important for the final RepairScope-X claim than adding more experiments on Qwen-origin failures.

Send back:

`RepairScopeX_Notebook11_Dream_Own_Failures_results.zip`